# TransUNet-RS — Training Demo

This notebook demonstrates the complete training pipeline for TransUNet-RS:
1. Model instantiation from config
2. Data loading and augmentation
3. Loss function and optimizer setup
4. Training loop (abbreviated)
5. Checkpoint saving

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import yaml
from src.models.transunet_rs import TransUNetRS
from src.training.loss import CombinedLoss
from src.training.optimizer import build_optimizer, build_scheduler

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Load Configuration

In [ ]:
with open('../configs/model_config.yaml', 'r') as f:
    model_cfg = yaml.safe_load(f)

with open('../configs/training_config.yaml', 'r') as f:
    train_cfg = yaml.safe_load(f)

print('Model config:')
for k, v in model_cfg.items():
    print(f'  {k}: {v}')

## 2. Instantiate Model

In [ ]:
model = TransUNetRS.from_config('../configs/model_config.yaml')

# Parameter counts
counts = model.count_parameters()
print('\nParameter counts:')
for name, count in counts.items():
    print(f'  {name:>15s}: {count:>12,}')

# Quick forward pass test
dummy_input = torch.randn(1, 3, 256, 256)
with torch.no_grad():
    output = model(dummy_input)
print(f'\nInput shape:  {dummy_input.shape}')
print(f'Output shape: {output.shape}')

## 3. Loss Function

In [ ]:
criterion = CombinedLoss(
    num_classes=10,
    ce_weight=0.5,
    dice_weight=0.5,
    label_smoothing=0.1,
)

# Test loss computation
dummy_logits = torch.randn(2, 10, 256, 256)
dummy_targets = torch.randint(0, 10, (2, 256, 256))
loss = criterion(dummy_logits, dummy_targets)
print(f'Combined Loss: {loss.item():.4f}')

## 4. Optimizer & Scheduler

In [ ]:
optimizer = build_optimizer(model, lr=1e-4, weight_decay=1e-4)
scheduler = build_scheduler(optimizer, T_max=100, warmup_epochs=5)

print(f'Optimizer: {type(optimizer).__name__}')
print(f'Initial LR: {optimizer.param_groups[0]["lr"]}')
print(f'\nParam groups:')
for i, group in enumerate(optimizer.param_groups):
    n_params = sum(p.numel() for p in group['params'])
    print(f'  Group {i}: {n_params:,} params, wd={group["weight_decay"]}')

## 5. Training Loop (Demo — 2 steps)

For full training, use: `python -m src.training.train --config configs/training_config.yaml`

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
criterion = criterion.to(device)

model.train()

for step in range(2):
    # Synthetic batch
    images = torch.randn(2, 3, 256, 256).to(device)
    masks = torch.randint(0, 10, (2, 256, 256)).to(device)
    
    optimizer.zero_grad()
    logits = model(images)
    loss = criterion(logits, masks)
    loss.backward()
    optimizer.step()
    
    print(f'Step {step+1}/2 — Loss: {loss.item():.4f}')

print('\nTraining demo complete!')